In [2]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from torchvision import transforms
import torchreid
from collections import deque

# -------------------- CONFIG --------------------
VIDEO_PATH = r"c:\Users\KIIT0001\Desktop\IIT BBSR\tracker\Screen Recording 2025-08-15 193525.mp4"
OUT_PATH   = r"c:\Users\KIIT0001\Desktop\IIT BBSR\tracker\videot1.mp4"

CONF_THRESHOLD = 0.6  # Higher confidence to avoid mislabels
REID_GALLERY_SIZE = 30
SIM_THRESHOLD   = 0.74
ALPHA_SIM_IOU   = 0.8
IOU_BONUS_THRESH= 0.30
STALE_KEEP_FRAMES = 999999
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# ------------------------------------------------

# 1) Detector (YOLOv8 - COCO model)
detector = YOLO("yolov8m.pt")
COCO_NAMES = detector.model.names  # YOLO's class name mapping

# 2) ReID model (still using OSNet — works for basic appearance matching)
reid_model = torchreid.models.build_model(
    name='osnet_x0_25', num_classes=1000, pretrained=True
).to(DEVICE).eval()

# 3) Preprocess for ReID
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

@torch.no_grad()
def get_embeddings_batch(crops):
    if not crops:
        return []
    tensors = []
    for img in crops:
        if img.size == 0:
            tensors.append(torch.zeros(3, 256, 128))
        else:
            tensors.append(preprocess(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
    batch = torch.stack(tensors).to(DEVICE)
    feats = reid_model(batch)
    feats = torch.nn.functional.normalize(feats, p=2, dim=1)
    return feats.cpu().numpy()

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0, inter_x2 - inter_x1)
    ih = max(0, inter_y2 - inter_y1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, ay2 - ay1)
    union = area_a + area_b - inter + 1e-6
    return inter / union

class IDEntry:
    def __init__(self, gid, emb, bbox, frame_idx):
        self.gid = gid
        self.embeds = deque([emb], maxlen=REID_GALLERY_SIZE)
        self.rep = emb.copy()
        self.bbox = bbox
        self.last_seen = frame_idx

    def update(self, emb, bbox, frame_idx, momentum=0.2):
        self.embeds.append(emb)
        self.rep = (1 - momentum) * self.rep + momentum * emb
        self.rep = self.rep / (np.linalg.norm(self.rep) + 1e-12)
        self.bbox = bbox
        self.last_seen = frame_idx

gallery = []
next_gid = 0

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Cannot open video: {VIDEO_PATH}"
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_writer = cv2.VideoWriter(
    OUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height)
)

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---------- Detect cars ----------
    yolo_res = detector(frame, verbose=False)[0]
    det_boxes_xyxy = []
    det_crops = []

    if hasattr(yolo_res, "boxes") and yolo_res.boxes is not None:
        for b in yolo_res.boxes.data.cpu().numpy():
            x1, y1, x2, y2, conf, cls = b
            label = COCO_NAMES[int(cls)]

            # Keep only cars with high confidence
            if label != "car" or conf < CONF_THRESHOLD:
                continue

            x1, y1 = max(0, int(x1)), max(0, int(y1))
            x2, y2 = min(width-1, int(x2)), min(height-1, int(y2))
            if x2 <= x1 or y2 <= y1:
                continue

            det_boxes_xyxy.append([x1, y1, x2, y2])
            det_crops.append(frame[y1:y2, x1:x2])

    # ---------- Embeddings ----------
    det_embs = get_embeddings_batch(det_crops)

    assigned = [-1] * len(det_boxes_xyxy)
    used_gids = set()

    for di, (box, emb) in enumerate(zip(det_boxes_xyxy, det_embs)):
        best_gid = -1
        best_score = -1.0
        for entry in gallery:
            if entry.gid in used_gids:
                continue
            sim = float(np.dot(emb, entry.rep))
            iou = iou_xyxy(box, entry.bbox) if entry.bbox is not None else 0.0
            score = ALPHA_SIM_IOU * sim + (1 - ALPHA_SIM_IOU) * iou
            thresh = SIM_THRESHOLD - 0.05 if iou > IOU_BONUS_THRESH else SIM_THRESHOLD
            if sim >= thresh and score > best_score:
                best_score = score
                best_gid = entry.gid
        if best_gid != -1:
            assigned[di] = best_gid
            used_gids.add(best_gid)

    for di, (box, emb) in enumerate(zip(det_boxes_xyxy, det_embs)):
        if assigned[di] == -1:
            gallery.append(IDEntry(next_gid, emb, box, frame_idx))
            assigned[di] = next_gid
            next_gid += 1
        else:
            gid = assigned[di]
            for entry in gallery:
                if entry.gid == gid:
                    entry.update(emb, box, frame_idx)
                    break

    # ---------- Draw ----------
    for (box, gid) in zip(det_boxes_xyxy, assigned):
        x1, y1, x2, y2 = box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(frame, f"Car ID {gid}", (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    out_writer.write(frame)
    cv2.imshow("YOLO + ReID (Cars only)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_idx += 1

cap.release()
out_writer.release()
cv2.destroyAllWindows()
print(f"Saved: {OUT_PATH}")


Successfully loaded imagenet pretrained weights from "C:\Users\KIIT0001/.cache\torch\checkpoints\osnet_x0_25_imagenet.pth"
Saved: c:\Users\KIIT0001\Desktop\IIT BBSR\tracker\videot1.mp4
